## RAG

R - retrieval  

A - augmented 

G - generation 

* First the document is stored in vector database.
* We take prompt from the user 
* Relevant information is **Retrieved** from the vector database on the basis of the prompt.
* The prompt and the context from the retrieved document is **Augmented**.
* This augmented prompt is used to **Generate** the response from LLM.

In [1]:
!pip install faiss -cpu

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: C:\Program Files\Python311\python.exe -m pip install --upgrade pip
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'pu'


In [2]:
pip install langchain-community 


Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install sentence-transformers

Note: you may need to restart the kernel to use updated packages.


In [25]:
# Import the libraries 

import os 
import google.generativeai as genai
from langchain_community.embeddings import HuggingFaceEmbeddings

from pypdf import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter

import faiss
from langchain_community.vectorstores import FAISS

In [24]:
pip install faiss-cpu

   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -----------------

In [47]:
# STEP 1: Configure the models 

# LLM mOdels
gemini_key = os.getenv("test-project-2")
genai.configure(api_key=gemini_key)
model = genai.GenerativeModel('gemini-2.5-flash')

# Configure Embedding model
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")




In [27]:
# STEP 2: Get the documents and process them 

pdf_file = PdfReader(r"sample.pdf")


raw_text = ""
for page in pdf_file.pages:
    text = page.extract_text()
    if text:
        raw_text += text +'\n'
        

In [28]:
print(raw_text)

1  
MAJOR PROJECT REPORT 
at 
Sathyabama Institute of Science and Technology 
(Deemed to be University) 
 
Submitted in partial fulfillment of the requirements for the award of 
Bachelor of Engineering Degree in Computer Science and Engineering 
 
By 
Busupalli Harinath Reddy(Reg.No.38110063) 
Avala Pavan Kumar (Reg. No.38110058) 
 
 
 
 
 
 
 
 
 
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING 
SCHOOL OF COMPUTING 
SATHYABAMA INSTITUTE OF SCIENCE AND TECHNOLOGY 
JEPPIAAR NAGAR, RAJIV GANDHI SALAI, 
CHENNAI – 600119, TAMILNADU 
 
 
MARCH 2022 

2  
 
SATHYABAMA 
INSTITUTE OF SCIENCE AND TECHNOLOGY 
(DEEMED TO BE UNIVERSITY) 
Accredited with Grade “A” by NAAC 
(Established under Section 3 of UGC Act, 1956) 
JEPPIAAR NAGAR, RAJIV GANDHI SALAI, CHENNAI– 600119 
www.sathyabamauniversity.ac.in 
 
 
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING 
 
 
                                               BONAFIDE CERTIFICATE 
 
 
This is to certify that this Project Report is the bonafide work of Av

In [29]:
## STEP 3: Chunking the text
# First we need to split the text 

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
chunks = splitter.split_text(raw_text)


In [30]:
len(raw_text)

37306

In [31]:
len(chunks)

49

In [32]:
len(chunks[0])

548

In [33]:
# STEP 4: Create the vector database

vector_store = FAISS.from_texts(chunks,embedding_model)

In [34]:
# STEP 5: Get the prompt from the user 
prompt = 'Give the brief introduction of the authors of this report.'

In [35]:
# STEP 6: Retrieval (R)

retriever = vector_store.as_retriever(search_kwargs={'k':3})
retrieved_docs = retriever.invoke(prompt)

In [42]:
retrieved_docs

[Document(id='a2761752-8d94-4912-a899-39349cbb72c2', metadata={}, page_content='Dr.S.Vigneshwari M.E., Ph.D., and Dr.L.Lakshmanan M.E., Ph.D., Heads of the  \nDepartment of Computer Science and Engineering for providing me necessary \nsupport and details at the right time during the progressive reviews. \nI would like to express my sincere and deep sense of gratitude to my Project Guide  Dr. \nR. AROUL CANESSANE  M.E., Ph.D.,  for her valuable guidance, suggestions and  \nconstant encouragement paved way for the successful completion of my project work. \nI wish to express my thanks to all Teaching and Non -teaching staff members of the  \nDepartment of Computer Science and Engineering who were helpful in many \nways for the completion of the project.\n5  \n \n                                 TABLE OF CONTENT \n \nINDEX \nNO \n                                             TITLE PAGE \nNO \n1.                                 ABSTRACT 6 \n2. INTRODUCTION 7 \n3.                            

In [43]:
context = "\n".join([d.page_content for d in retrieved_docs])

In [44]:
print(context)

Dr.S.Vigneshwari M.E., Ph.D., and Dr.L.Lakshmanan M.E., Ph.D., Heads of the  
Department of Computer Science and Engineering for providing me necessary 
support and details at the right time during the progressive reviews. 
I would like to express my sincere and deep sense of gratitude to my Project Guide  Dr. 
R. AROUL CANESSANE  M.E., Ph.D.,  for her valuable guidance, suggestions and  
constant encouragement paved way for the successful completion of my project work. 
I wish to express my thanks to all Teaching and Non -teaching staff members of the  
Department of Computer Science and Engineering who were helpful in many 
ways for the completion of the project.
5  
 
                                 TABLE OF CONTENT 
 
INDEX 
NO 
                                             TITLE PAGE 
NO 
1.                                 ABSTRACT 6 
2. INTRODUCTION 7 
3.                                     AIM 13 
4.                                   SCOPE 13
2  
 
SATHYABAMA 
INSTITUTE OF SCIEN

In [45]:
# STEP 7: Augmenting (A)

augmented_prompt = f'''
<role> You are helpful assistant using RAG
<goal> Answer the question asked by the user based on the context provided. {prompt}
<context> Here are the documents retrieved from the vector database to support the answer which you have to generate {context}>
'''

In [48]:
# STEP 8: 


response = model.generate_content(augmented_prompt)
print(response.text)

Based on the provided context, the authors and key contributors to this project report are:

*   **Avala Pavan Kumar** (Registration Number: 38110058)
*   **Busupalli Harinath Reddy** (Registration Number: 38110063)

They are the students who carried out the project entitled "A SYSTEMATIC APPROACH TOWARDS DESCRIPTION AND CLASSIFICATION OF CRIME INCIDENTS" as part of their Bachelor of Engineering degree in Computer Science and Engineering at Sathyabama Institute of Science and Technology.

The report also acknowledges the significant contributions of:

*   **Dr. R. AROUL CANESSANE M.E., Ph.D.,** who served as the Project Guide (Internal Guide) for the students, providing valuable guidance, suggestions, and constant encouragement.
*   **Dr. S.VIGNESHWARI M.E., Ph.D.,** who is the Head of the Department of Computer Science and Engineering. She provided necessary support and details during the project's reviews.
*   **Dr. L. LAKSHMANAN M.E., Ph.D.,** also a Head of the Department of Comput